In [21]:
# PyTorch로 텐서를 만들고 신경망을 학습
import torch
from torch.utils.data import Dataset, DataLoader

# 학습용 영어–한국어 문장 쌍 5개임 실제 번역 성능을 평가하기 위한 데이터는 아님
# 수정부분 데이터셋 변경
pairs = [
    ("i love you very much", "나는 너를 매우 사랑해"),
    ("he is happy every day", "그는 매일 행복하다"),
    ("she is a good student", "그녀는 좋은 학생이다"),
    ("thank you for your help", "도와줘서 고마워"),
    ("good morning to you all", "모두 좋은 아침"),

    ("i am a good student", "나는 좋은 학생이다"),
    ("you are happy every day", "너는 매일 행복하다"),
    ("he loves you very much", "그는 너를 매우 사랑한다"),
    ("she loves me very much", "그녀는 나를 매우 사랑한다"),
    ("i like music very much", "나는 음악을 매우 좋아한다"),

    ("i like apples very much", "나는 사과를 매우 좋아한다"),
    ("you like music every day", "너는 매일 음악을 좋아한다"),
    ("he likes apples very much", "그는 사과를 매우 좋아한다"),
    ("she is happy every day", "그녀는 매일 행복하다"),
    ("i am happy every day", "나는 매일 행복하다")
]

# 간단한 토크나이저
# 공백을 기준으로 분리 예: "i love you" → ["i", "love", "you"].
# 한국어도 공백 단위로 나누며, 형태소 분석은 미수행
def tokenize(sentence):
    return sentence.split()

# 단어 사전 구성
# 영어 사전과 한국어 사전을 따로 생성 단어를 정수 ID로 변환하기 위한 표임
# <pad>: 길이 맞춤용 빈칸, <sos>: 문장 시작, <eos>: 문장 끝
src_vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2}
tgt_vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2}

# 모든 문장을 읽으며 처음 보는 단어에 현재 사전 크기를 ID로 부여
for src, tgt in pairs:
    for w in tokenize(src):
        if w not in src_vocab:
            src_vocab[w] = len(src_vocab)
    for w in tokenize(tgt):
        if w not in tgt_vocab:
            tgt_vocab[w] = len(tgt_vocab)
# 결과값 확인용
for i in src_vocab:
    print(i, src_vocab[i])
for i in tgt_vocab:
    print(i, tgt_vocab[i])

# 한국어 ID → 단어의 역방향 사전임 예측한 ID를 사람이 읽는 단어로 복원
inv_tgt = {v:k for k,v in tgt_vocab.items()}

# 시작·종료 토큰까지 포함하여 모든 문장을 8개 토큰으로 조정
max_len = 12

def encode(sentence, vocab):
    # 문장 앞뒤에 시작·종료 토큰을 추가
    tokens = ["<sos>"] + tokenize(sentence) + ["<eos>"]
    # 길면 앞의 8개만 유지하고, 짧으면 뒤에 <pad>를 채움
    # 긴 문장에서는 <eos>도 잘릴 수 있음 현재 예제 문장은 모두 길이 제한 안에 있음
    tokens = tokens[:max_len] + ["<pad>"] * (max_len - len(tokens))
    # 각 토큰을 사전 ID로 변환 결과는 길이 8의 정수 텐서임
    # <unk>가 없으므로 사전에 없는 단어가 들어오면 KeyError가 발생
    return torch.tensor([vocab[t] for t in tokens])

# Dataset은 DataLoader가 문장 쌍을 하나씩 가져갈 수 있도록 하는 인터페이스임
class ToyTranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    # 데이터 개수를 반환 여기서는 5임
    def __len__(self):
        return len(self.pairs)

    # idx번째 문장 쌍을 가져와 영어·한국어를 각각 ID 텐서로 반환
    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_encoded = encode(src, src_vocab)     # shape: (max_len,)
        tgt_encoded = encode(tgt, tgt_vocab)     # shape: (max_len,)
        return src_encoded, tgt_encoded

dataset = ToyTranslationDataset(pairs)
# 한 번에 최대 2개의 문장 쌍을 묶고, 매 epoch마다 순서를 섞음
# 배치 텐서 크기: (B, 8) 데이터가 5개이므로 마지막 배치의 B는 1임
loader = DataLoader(dataset, batch_size=2, shuffle=True)



<pad> 0
<sos> 1
<eos> 2
i 3
love 4
you 5
very 6
much 7
he 8
is 9
happy 10
every 11
day 12
she 13
a 14
good 15
student 16
thank 17
for 18
your 19
help 20
morning 21
to 22
all 23
am 24
are 25
loves 26
me 27
like 28
music 29
apples 30
likes 31
<pad> 0
<sos> 1
<eos> 2
나는 3
너를 4
매우 5
사랑해 6
그는 7
매일 8
행복하다 9
그녀는 10
좋은 11
학생이다 12
도와줘서 13
고마워 14
모두 15
아침 16
너는 17
사랑한다 18
나를 19
음악을 20
좋아한다 21
사과를 22


In [22]:
import torch.nn as nn
import torch.nn.functional as F

# Bahdanau(가산) 어텐션: 디코더의 상태를 기준으로 입력의 각 위치가 얼마나 중요한지 계산
# 기호: B=배치 크기, T=입력 길이, H=은닉 벡터 크기
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        # 인코더의 각 위치 벡터를 어텐션 계산용 공간으로 변환
        self.W1 = nn.Linear(hidden_size, hidden_size)
        # 디코더의 은닉 벡터를 같은 크기의 공간으로 변환
        self.W2 = nn.Linear(hidden_size, hidden_size)
        # 결합된 H차원 벡터를 위치별 중요도 점수 1개로 변환
        self.V  = nn.Linear(hidden_size, 1)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: (B, H)
        # encoder_outputs: (B, T, H)
        # score: (B, T, 1)
        # W1(encoder_outputs): (B,T,H), W2(decoder_hidden): (B,H)
        # unsqueeze(1)로 (B,1,H)를 만들면 입력의 T개 위치에 방송(broadcast)하여 더할 수 있음
        # tanh를 거친 뒤 V를 적용하면 각 위치의 점수 (B,T,1)가 됨
        score = self.V(torch.tanh(
            self.W1(encoder_outputs) + self.W2(decoder_hidden).unsqueeze(1)
        ))
        # dim=1은 입력 위치 축임 각 문장의 T개 가중치 합이 1이 되도록 softmax를 적용
        # 이 예제는 패딩 마스크가 없어 <pad> 위치도 어텐션에 포함됨
        attn_weights = F.softmax(score, dim=1)         # (B, T, 1)
        # 각 입력 벡터에 가중치를 곱한 뒤 위치 축으로 합산
        # 예: 가중치가 [0.7, 0.2, 0.1]이면 context = 0.7*h1 + 0.2*h2 + 0.1*h3
        context = (attn_weights * encoder_outputs).sum(dim=1)   # (B, H)
        return context, attn_weights


In [23]:
# 인코더는 영어 문장을 읽고 각 위치의 특징과 마지막 LSTM 상태를 생성
# B=배치 크기, T=문장 길이, E=임베딩 크기, H=은닉 크기, V=한국어 사전 크기
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__() # nn.Module 초기화
        # 단어 ID 하나를 학습 가능한 E차원 벡터로 변환
        self.embed = nn.Embedding(vocab_size, embed_size)
        # batch_first=True이므로 입력 순서는 (배치, 시간, 특징)임
        self.lstm  = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        # x: (B, T)
        # (B,T)의 정수 ID를 (B,T,E)의 실수 벡터로 변환
        x = self.embed(x)                 # (B, T, E)
        # outputs는 모든 위치의 은닉 상태임 h와 c는 마지막 은닉 상태와 셀 상태임
        # 현재 단층·단방향 LSTM에서 h, c의 크기는 각각 (1,B,H)임
        # 입력 전체를 처리하므로 여기서 마지막 상태에는 뒤쪽 <pad> 처리도 반영됨
        outputs, (h, c) = self.lstm(x)    # outputs: (B, T, H)
        return outputs, (h, c)

# 디코더는 토큰 하나와 어텐션 문맥을 받아 다음 한국어 토큰을 예측합니다
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, attention):
        super().__init__()
        # 단어 ID 하나를 학습 가능한 E차원 벡터로 변환
        self.embed = nn.Embedding(vocab_size, embed_size)
        # 임베딩 E차원과 문맥 H차원을 이어 붙이므로 입력 특징 크기는 E+H임
        self.lstm  = nn.LSTM(embed_size + hidden_size, hidden_size, batch_first=True)
        self.attn  = attention
        # LSTM의 H차원 출력을 한국어 사전의 V개 점수로 변환
        self.fc    = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        # x: (B,) 단일 토큰
        # 토큰 ID (B,) → 임베딩 (B,E) → 시간 축을 추가한 (B,1,E)
        # 한 번의 호출에서 시간 단계 하나만 처리
        x = self.embed(x).unsqueeze(1)      # (B,1,E)

        # 디코더 현재 은닉 상태
        # hidden[-1]은 마지막 층의 은닉 상태임 (1,B,H)에서 (B,H)를 추출
        decoder_hidden = hidden[-1]         # (B,H)

        # 어텐션 계산
        # 직전 디코더 상태를 질의로 사용하여 입력 문장에서 참고할 정보를 수집
        # 첫 단계에서는 인코더의 마지막 은닉 상태를 사용 _는 사용하지 않는 어텐션 가중치임
        context, _ = self.attn(decoder_hidden, encoder_outputs)  # (B,H)
        # 문맥 (B,H)에 시간 축을 추가하여 임베딩과 결합
        context = context.unsqueeze(1)       # (B,1,H)

        # LSTM 입력 결합
        # dim=2는 특징 축임 (B,1,E)와 (B,1,H)를 이어 붙여 (B,1,E+H)를 생성
        lstm_input = torch.cat([x, context], dim=2)  # (B,1,E+H)

        # 이전 hidden, cell을 받아 한 단계 진행하고 갱신된 h, c를 반환
        outputs, (h, c) = self.lstm(lstm_input, (hidden, cell))  # outputs: (B,1,H)
        # squeeze(1)은 크기가 1인 시간 축만 제거: (B,1,H) → (B,H)
        # logits는 확률 변환 전의 V개 점수이며, CrossEntropyLoss에 그대로 전달
        logits = self.fc(outputs.squeeze(1))                     # (B,V)
        return logits, (h, c)

# 인코더와 디코더를 연결한 전체 번역 모델임
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        # 영어 문장을 한 번 인코딩하고, 마지막 h와 c로 디코더를 초기화
        encoder_outputs, (h, c) = self.encoder(src)

        batch_size = src.size(0)
        # 여기서 T는 정답 문장의 길이임 이 예제에서는 입력·정답 모두 길이 8임
        T = tgt.size(1)

        outputs = []
        # 모든 문장의 첫 토큰 <sos>를 선택 크기는 (B,)임
        token = tgt[:, 0]

        # <sos> 다음부터 마지막 위치까지 총 T-1번 예측
        for t in range(1, T):
            logits, (h, c) = self.decoder(token, h, c, encoder_outputs)
            # (B,V)에 시간 축을 추가한 (B,1,V)를 리스트에 저장
            outputs.append(logits.unsqueeze(1))
            # Teacher forcing: 예측 토큰 대신 현재 위치의 실제 정답을 다음 단계의 입력으로 사용
            # 예: <sos>로 "나는"을 예측한 다음, 정답 "나는"을 입력하여 "너를"을 예측
            token = tgt[:, t]     # teacher forcing

        # 시간 축으로 이어 붙여 (B,T-1,V)를 반환 각 위치에는 다음 토큰의 점수가 있음
        return torch.cat(outputs, dim=1)   # (B, T-1, V)


In [24]:
# 단어 하나를 표현하는 벡터의 차원 E=32임
embed_size = 32
# LSTM 상태와 어텐션 문맥의 차원 H=64임
hidden_size = 128

# 같은 은닉 크기를 사용하여 어텐션, 인코더, 디코더를 구성하고 전체 모델로 연결
attention = BahdanauAttention(hidden_size)
encoder    = Encoder(len(src_vocab), embed_size, hidden_size)
decoder    = Decoder(len(tgt_vocab), embed_size, hidden_size, attention)
model      = Seq2Seq(encoder, decoder)

# 정답 토큰의 점수가 높아지도록 교차 엔트로피 손실을 최소화
# <pad> 정답은 손실에서 제외 <eos>는 학습 대상임
# 손실에서 패딩을 제외하는 것과 어텐션에서 패딩을 가리는 것은 별개의 처리임
criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab["<pad>"])
# Adam이 모델의 모든 학습 가능한 파라미터를 학습률 0.001로 갱신
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [27]:
# 전체 학습 데이터를 40번 반복해서 학습
num_epochs = 40

for epoch in range(num_epochs):
    # 현재 epoch의 배치별 손실을 합산할 변수임
    total_loss = 0.0

    # 영어와 한국어 ID 텐서 배치를 불러오기 각각 (B,8)임
    for src, tgt in loader:
        # 이전 배치의 기울기를 초기화 PyTorch의 기울기는 기본적으로 누적되기 때문
        optimizer.zero_grad()

        # 정답을 입력으로 쓰는 teacher forcing으로 (B,7,V)의 예측 점수를 획득
        logits = model(src, tgt)

        # 모든 배치·시간 위치를 펼쳐 토큰 분류 문제로 계산
        # 예측 점수: (B,7,V) → (B*7,V), 정답: (B,7) → (B*7,)
        # tgt[:,1:]은 <sos>를 제외한 다음 토큰 정답이며 예측 위치와 일치
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt[:, 1:].reshape(-1)
        )

        # 손실에서 각 파라미터로 역전파하여 기울기를 계산
        loss.backward()
        # 계산된 기울기를 이용해 파라미터를 갱신
        optimizer.step()

        # loss.item()은 스칼라 텐서를 Python 숫자로 추출
        total_loss += loss.item()

    # 표시하는 Loss는 배치별 평균 손실의 합임 전체 토큰에 대한 평균은 아님
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 1.7727
Epoch 2: Loss = 1.6836
Epoch 3: Loss = 1.4375
Epoch 4: Loss = 1.1605
Epoch 5: Loss = 1.1269
Epoch 6: Loss = 1.0000
Epoch 7: Loss = 0.9315
Epoch 8: Loss = 0.8666
Epoch 9: Loss = 0.8241
Epoch 10: Loss = 0.7934
Epoch 11: Loss = 0.6403
Epoch 12: Loss = 0.5926
Epoch 13: Loss = 0.5479
Epoch 14: Loss = 0.4857
Epoch 15: Loss = 0.4550
Epoch 16: Loss = 0.4292
Epoch 17: Loss = 0.4025
Epoch 18: Loss = 0.3934
Epoch 19: Loss = 0.3742
Epoch 20: Loss = 0.3640
Epoch 21: Loss = 0.3282
Epoch 22: Loss = 0.3149
Epoch 23: Loss = 0.3087
Epoch 24: Loss = 0.2854
Epoch 25: Loss = 0.2788
Epoch 26: Loss = 0.2776
Epoch 27: Loss = 0.2546
Epoch 28: Loss = 0.2586
Epoch 29: Loss = 0.2491
Epoch 30: Loss = 0.2287
Epoch 31: Loss = 0.2212
Epoch 32: Loss = 0.2193
Epoch 33: Loss = 0.2104
Epoch 34: Loss = 0.2085
Epoch 35: Loss = 0.1990
Epoch 36: Loss = 0.1879
Epoch 37: Loss = 0.1815
Epoch 38: Loss = 0.1814
Epoch 39: Loss = 0.1691
Epoch 40: Loss = 0.1640


In [28]:
# 추론에서는 정답 문장 없이 이전에 예측한 토큰을 입력으로 사용
# 현재 코드는 eval()/no_grad()를 사용하지 않으므로 추론 시에도 연산 그래프가 만들어짐
# 실용적인 추론 코드에서는 model.eval()과 torch.no_grad() 사용을 권장
def translate(sentence):
    # 영어 문장 (8,)에 배치 축을 추가하여 (1,8)로 생성
    src = encode(sentence, src_vocab).unsqueeze(0)
    # 입력 특징을 구하고 인코더의 마지막 상태로 디코더를 초기화
    encoder_outputs, (h, c) = encoder(src)

    # 첫 입력은 한국어 문장 시작 토큰 <sos> 하나임 크기는 (1,)임
    token = torch.tensor([tgt_vocab["<sos>"]])
    result = []

    # 종료 토큰이 나오지 않아도 최대 8단계까지만 생성
    for _ in range(max_len):
        # 현재 토큰과 상태로 다음 토큰 점수를 계산하고 LSTM 상태를 갱신
        logits, (h, c) = decoder(
            token, h, c, encoder_outputs
        )

        # V개 점수 중 가장 큰 점수의 토큰 ID를 선택하는 탐욕적(greedy) 생성임
        # 학습 때와 달리 이 예측 ID 자체를 다음 단계 입력으로 사용
        token = logits.argmax(dim=1)
        # 배치 크기가 1이므로 item()으로 ID 하나를 추출하여 한국어 단어로 복원
        word = inv_tgt[token.item()]

        # <eos>를 예측하면 번역을 종료하며 종료 토큰 자체는 결과에 넣지 않음
        if word == "<eos>":
            break

        # 생성한 단어를 순서대로 수집 이 예제는 <pad>/<sos>가 예측되면 그대로 추가
        result.append(word)

    # 단어들을 공백으로 이어 붙여 번역 문자열을 반환
    return " ".join(result)

# 학습에 사용한 문장으로 동작을 확인 미지의 문장에 대한 일반화 평가가 아님
# 초기값과 데이터 순서가 무작위이므로 40 epoch 후에도 정답 번역을 보장되지 않음
for src, tgt in pairs:
    pred = translate(src)

    print(f"영어 원문 : {src}")
    print(f"실제 정답 : {tgt}")
    print(f"번역 결과 : {pred}")
    print(f"일치 여부 : {'O' if pred == tgt else 'X'}")
    print("-" * 40)


영어 원문 : i love you very much
실제 정답 : 나는 너를 매우 사랑해
번역 결과 : 나는 너를 매우 사랑해
일치 여부 : O
----------------------------------------
영어 원문 : he is happy every day
실제 정답 : 그는 매일 행복하다
번역 결과 : 그는 매일 행복하다
일치 여부 : O
----------------------------------------
영어 원문 : she is a good student
실제 정답 : 그녀는 좋은 학생이다
번역 결과 : 그녀는 좋은 학생이다
일치 여부 : O
----------------------------------------
영어 원문 : thank you for your help
실제 정답 : 도와줘서 고마워
번역 결과 : 도와줘서 고마워
일치 여부 : O
----------------------------------------
영어 원문 : good morning to you all
실제 정답 : 모두 좋은 아침
번역 결과 : 모두 좋은 아침
일치 여부 : O
----------------------------------------
영어 원문 : i am a good student
실제 정답 : 나는 좋은 학생이다
번역 결과 : 나는 좋은 학생이다
일치 여부 : O
----------------------------------------
영어 원문 : you are happy every day
실제 정답 : 너는 매일 행복하다
번역 결과 : 너는 매일 행복하다
일치 여부 : O
----------------------------------------
영어 원문 : he loves you very much
실제 정답 : 그는 너를 매우 사랑한다
번역 결과 : 그는 너를 매우 사랑한다
일치 여부 : O
----------------------------------------
영어 원문 : she loves me very much
실제 정답 : 그

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(23)

B, T, H = 1, 3, 4

decoder_hidden = torch.randn(B, H)
encoder_outputs = torch.randn(B, T, H)

W1 = nn.Linear(H, H)
W2 = nn.Linear(H, H)
V = nn.Linear(H, 1)

a = W1(encoder_outputs)
b = W2(decoder_hidden)
b = b.unsqueeze(1)

score = V(torch.tanh(a + b))
attn_weights = F.softmax(score, dim=1)
context = (attn_weights * encoder_outputs).sum(dim=1)

print("encoder_outputs:", encoder_outputs.shape)
print("decoder_hidden:", decoder_hidden.shape)
print("W1 결과:", a.shape)
print("W2 + unsqueeze:", b.shape)
print("score:", score.shape)
print("attn_weights:", attn_weights.shape)
print("context:", context.shape)

print("\n단어별 어텐션 가중치:")
for word, weight in zip(
    ["I", "love", "you"],
    attn_weights[0, :, 0].tolist()
):
    print(f"{word}: {weight:.4f}")


encoder_outputs: torch.Size([1, 3, 4])
decoder_hidden: torch.Size([1, 4])
W1 결과: torch.Size([1, 3, 4])
W2 + unsqueeze: torch.Size([1, 1, 4])
score: torch.Size([1, 3, 1])
attn_weights: torch.Size([1, 3, 1])
context: torch.Size([1, 4])

단어별 어텐션 가중치:
I: 0.3631
love: 0.3130
you: 0.3238
